In [1]:
import subprocess
from pathlib import Path
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach", commit],
    check=True,
)

print("원본 코드 커밋:",
      subprocess.check_output(
          ["git", "-C", str(repo), "rev-parse", "HEAD"],
          text=True,
      ).strip())
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

원본 코드 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

log_dir = Path("/content/TCN/reproduction_logs")
log_dir.mkdir(parents=True, exist_ok=True)
copy_log = log_dir / (
    f"copy_memory_T1000_seed1111_10epochs_{datetime.now():%Y%m%d_%H%M%S}.log"
)

runner = """
import numpy as np
import runpy
import sys

np.random.seed(1111)
sys.argv = [
    "copymem_test",
    "--blank_len", "1000",
    "--seq_len", "10",
    "--ksize", "8",
    "--levels", "8",
    "--nhid", "10",
    "--dropout", "0.05",
    "--epochs", "10",
    "--seed", "1111",
]
runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")
"""

print("Copy Memory / T=1000 / 시드 1111 / 10에폭")
print("로그 저장 위치:", copy_log)
started = time.perf_counter()

with copy_log.open("w") as log_file:
    process = subprocess.Popen(
        [sys.executable, "-u", "-c", runner],
        cwd="/content/TCN",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
print("로그 파일:", copy_log)

if return_code != 0:
    raise RuntimeError("Copy Memory 실행에 실패했습니다.")

Copy Memory / T=1000 / 시드 1111 / 10에폭
로그 저장 위치: /content/TCN/reproduction_logs/copy_memory_T1000_seed1111_10epochs_20261124_050627.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=10, ksize=8, iters=100, levels=8, blank_len=1000, seq_len=10, log_interval=50, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
| Epoch   1 |    50/  313 batches | lr 0.00050 | ms/batch 47.01 | loss 0.24876210 | accuracy 94.2034
| Epoch   1 |   100/  313 batches | lr 0.00050 | ms/batch  9.98 | loss 0.05640970 | accuracy 98.9066
| Epoch   1 |   150/  313 batches | lr 0.00050 | ms/batch 10.16 | loss 0.03764843 | accuracy 99.0191
| Epoch   1 |   200/  313 batches | lr 0.00050 | ms/batch  9.88 | loss 0.02781480 | accuracy 99.0646
| Epoch   1 |   250

In [3]:
from google.colab import files

files.download(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_10epochs_20261124_050627.log"
)

In [4]:
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

log_dir = Path("/content/TCN/reproduction_logs")
log_dir.mkdir(parents=True, exist_ok=True)
copy50_log = log_dir / (
    f"copy_memory_T1000_seed1111_50epochs_{datetime.now():%Y%m%d_%H%M%S}.log"
)

runner = """
import numpy as np
import runpy
import sys
import torch
import torch.nn.functional as F
from pathlib import Path

np.random.seed(1111)
sys.argv = [
    "copymem_test",
    "--blank_len", "1000",
    "--seq_len", "10",
    "--ksize", "8",
    "--levels", "8",
    "--nhid", "10",
    "--dropout", "0.05",
    "--epochs", "50",
    "--seed", "1111",
    "--log-interval", "1000",
]

state = runpy.run_module(
    "TCN.copy_memory.copymem_test",
    run_name="__main__",
)

model = state["model"]
test_x = state["test_x"]
test_y = state["test_y"]
model.eval()

with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    predictions = logits.argmax(dim=-1)

    overall_accuracy = (
        (predictions == test_y).float().mean().item() * 100
    )
    final10_accuracy = (
        (predictions[:, -10:] == test_y[:, -10:])
        .float().mean().item() * 100
    )
    final10_loss = F.cross_entropy(
        logits[:, -10:, :].reshape(-1, 10),
        test_y[:, -10:].reshape(-1),
    ).item()

parameter_count = sum(p.numel() for p in model.parameters())

checkpoint = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_50epochs_checkpoint.pt"
)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "completed_epochs": 50,
        "upstream_commit": "2f8c2b817050206397458dfd1f5a25ce8a32fe65",
    },
    checkpoint,
)

print("FINAL_MODEL_PARAMETERS:", parameter_count)
print(f"FINAL_OVERALL_ACCURACY: {overall_accuracy:.4f}%")
print(f"FINAL_LAST10_ACCURACY: {final10_accuracy:.4f}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {final10_loss:.8f}")
print("CHECKPOINT:", checkpoint)
"""

print("Copy Memory / T=1000 / 시드 1111 / 처음부터 50에폭")
print("로그 저장 위치:", copy50_log)
started = time.perf_counter()

with copy50_log.open("w") as log_file:
    process = subprocess.Popen(
        [sys.executable, "-u", "-c", runner],
        cwd="/content/TCN",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
print("로그 파일:", copy50_log)

if return_code != 0:
    raise RuntimeError("50에폭 실행에 실패했습니다.")

Copy Memory / T=1000 / 시드 1111 / 처음부터 50에폭
로그 저장 위치: /content/TCN/reproduction_logs/copy_memory_T1000_seed1111_50epochs_20261124_051148.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=8, iters=100, levels=8, blank_len=1000, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.02149778  |  Accuracy: 99.1283


Test set: Average loss: 0.02003532  |  Accuracy: 99.1928


Test set: Average loss: 0.01248106  |  Accuracy: 99.4627


Test set: Average loss: 0.00843873  |  Accuracy: 99.7386


Test set: Average loss: 0.00724102  |  Accuracy: 99.8204


Test set: Average loss: 0.01021435  |  Accuracy: 99.3895


Test set: Average loss: 0.00756327  |  Accuracy: 99.6405


Te

In [5]:
import zipfile
from pathlib import Path
from google.colab import files

checkpoint = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_50epochs_checkpoint.pt"
)
backup_zip = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_backup_20261124.zip"
)

with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (copy_log, copy50_log, checkpoint):
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, path.name)

print("백업 파일:", backup_zip)
files.download(str(backup_zip))

백업 파일: /content/TCN/reproduction_logs/copy_memory_T1000_backup_20261124.zip


In [6]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

# 런타임이 재시작됐을 경우를 대비해 원본 커밋 확인
repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 2000
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
copy2000_log = log_dir / f"copy_memory_T{T}_seed1111_50epochs_{stamp}.log"
checkpoint = log_dir / f"copy_memory_T{T}_seed1111_50epochs_checkpoint.pt"

runner = f"""
import numpy as np, runpy, sys, torch
import torch.nn.functional as F

np.random.seed(1111)
sys.argv = ["copymem_test",
    "--blank_len", "{T}", "--seq_len", "10",
    "--ksize", "8", "--levels", "8", "--nhid", "10",
    "--dropout", "0.05", "--epochs", "50", "--seed", "1111",
    "--log-interval", "1000"]
state = runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")

model, test_x, test_y = state["model"], state["test_x"], state["test_y"]
model.eval()
with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    pred = logits.argmax(dim=-1)
    overall = (pred == test_y).float().mean().item() * 100
    last10 = (pred[:, -10:] == test_y[:, -10:]).float().mean().item() * 100
    last10_ce = F.cross_entropy(logits[:, -10:, :].reshape(-1, 10),
                                test_y[:, -10:].reshape(-1)).item()

torch.save({{"model_state_dict": model.state_dict(), "completed_epochs": 50,
            "blank_len": {T}, "upstream_commit": "{commit}"}}, "{checkpoint}")
print("SEQUENCE_TOTAL_LENGTH:", test_y.shape[1])
print("FINAL_MODEL_PARAMETERS:", sum(p.numel() for p in model.parameters()))
print(f"FINAL_OVERALL_ACCURACY: {{overall:.4f}}%")
print(f"FINAL_LAST10_ACCURACY: {{last10:.4f}}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {{last10_ce:.8f}}")
print("CHECKPOINT:", "{checkpoint}")
"""

print(f"Copy Memory / T={T} / 시드 1111 / 처음부터 50에폭")
print("로그:", copy2000_log)
started = time.perf_counter()
with copy2000_log.open("w") as log_file:
    process = subprocess.Popen([sys.executable, "-u", "-c", runner], cwd=str(repo),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line); log_file.flush()
return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
if return_code != 0:
    raise RuntimeError(f"T={T} 실행에 실패했습니다.")

# 로그 + 체크포인트 백업 ZIP 다운로드
backup_zip = log_dir / f"copy_memory_T{T}_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (copy2000_log, checkpoint):
        archive.write(path, path.name)
print("백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4
Copy Memory / T=2000 / 시드 1111 / 처음부터 50에폭
로그: /content/TCN/reproduction_logs/copy_memory_T2000_seed1111_50epochs_20261124_052742.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=8, iters=100, levels=8, blank_len=2000, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.01122813  |  Accuracy: 99.5602


Test set: Average loss: 0.01015619  |  Accuracy: 99.5853


Test set: Average loss: 0.00611462  |  Accuracy: 99.7341


Test set: Average loss: 0.00496229  |  Accuracy: 99.7697


Test set: Average loss: 0.00381203  |  Accuracy: 99.9168


Test set: Average loss: 0.00365342  |  Accuracy: 99.9344


Test 

In [7]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

script_path = Path("/content/run_copy_memory_recurrent.py")
script_path.write_text(r'''
"""Copy Memory recurrent baselines (GRU/LSTM) using the original TCN data generator."""
import argparse, json, sys, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

parser = argparse.ArgumentParser()
parser.add_argument("--repo", default="/content/TCN")
parser.add_argument("--model", choices=["GRU", "LSTM", "RNN"], required=True)
parser.add_argument("--hidden", type=int, required=True)
parser.add_argument("--blank_len", type=int, default=1000)
parser.add_argument("--seq_len", type=int, default=10)
parser.add_argument("--epochs", type=int, default=50)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--lr", type=float, default=5e-4)
parser.add_argument("--clip", type=float, default=1.0)
parser.add_argument("--seed", type=int, default=1111)
parser.add_argument("--out_prefix", required=True)
args = parser.parse_args()
print(args, flush=True)

sys.path.insert(0, args.repo)
from TCN.copy_memory.utils import data_generator

# TCN 실행과 같은 순서: torch 시드 → NumPy 시드로 학습·테스트 데이터 생성 → 모델 생성
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
np.random.seed(args.seed)
T, seq_len, n_classes = args.blank_len, args.seq_len, 10
train_x, train_y = data_generator(T, seq_len, 10000)
test_x, test_y = data_generator(T, seq_len, 1000)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_x, train_y, test_x, test_y = (t.to(device) for t in (train_x, train_y, test_x, test_y))


class RecurrentBaseline(nn.Module):
    def __init__(self, cell, hidden):
        super().__init__()
        self.rnn = getattr(nn, cell)(1, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, n_classes)

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return self.linear(out)


model = RecurrentBaseline(args.model, args.hidden).to(device)
params = sum(p.numel() for p in model.parameters())
print("MODEL_PARAMETERS:", params, flush=True)
optimizer = torch.optim.RMSprop(model.parameters(), lr=args.lr)


def evaluate():
    model.eval()
    with torch.no_grad():
        logits = model(test_x)
        loss = F.cross_entropy(logits.reshape(-1, n_classes), test_y.reshape(-1)).item()
        pred = logits.argmax(-1)
        overall = (pred == test_y).float().mean().item() * 100
        last10 = (pred[:, -seq_len:] == test_y[:, -seq_len:]).float().mean().item() * 100
        last10_ce = F.cross_entropy(logits[:, -seq_len:].reshape(-1, n_classes),
                                    test_y[:, -seq_len:].reshape(-1)).item()
    return loss, overall, last10, last10_ce


history = []
started = time.perf_counter()
for ep in range(1, args.epochs + 1):
    model.train()
    ep_start = time.perf_counter()
    for start in range(0, train_x.size(0), args.batch_size):
        x = train_x[start:start + args.batch_size]
        y = train_y[start:start + args.batch_size]
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x).reshape(-1, n_classes), y.reshape(-1))
        loss.backward()
        if args.clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)  # backward 뒤 적용
        optimizer.step()
    loss, overall, last10, last10_ce = evaluate()
    history.append({"epoch": ep, "test_loss": loss, "overall_accuracy": overall,
                    "last10_accuracy": last10, "last10_cross_entropy": last10_ce,
                    "epoch_seconds": time.perf_counter() - ep_start})
    print(f"\nTest set: Average loss: {loss:.8f}  |  Accuracy: {overall:.4f}", flush=True)
    print(f"| Epoch {ep:3d} | last10 accuracy {last10:.4f} | last10 CE {last10_ce:.8f} "
          f"| {time.perf_counter() - ep_start:.1f}s", flush=True)

final = history[-1]
print("FINAL_MODEL_PARAMETERS:", params)
print(f"FINAL_OVERALL_ACCURACY: {final['overall_accuracy']:.4f}%")
print(f"FINAL_LAST10_ACCURACY: {final['last10_accuracy']:.4f}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {final['last10_cross_entropy']:.8f}")
print(f"TRAIN_SECONDS: {time.perf_counter() - started:.1f}")
json.dump({"args": vars(args), "parameters": params, "history": history},
          open(args.out_prefix + "_results.json", "w"), indent=2)
torch.save({"model_state_dict": model.state_dict(), "args": vars(args),
            "upstream_commit": "2f8c2b817050206397458dfd1f5a25ce8a32fe65"},
           args.out_prefix + "_checkpoint.pt")
''')

log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs = [script_path]

for model_name, hidden in (("GRU", 61), ("LSTM", 53)):
    prefix = str(log_dir / f"copy_memory_T1000_{model_name.lower()}_seed1111_50epochs_{stamp}")
    log_path = Path(prefix + ".log")
    print(f"\n===== Copy Memory / T=1000 / {model_name} (hidden {hidden}) / 시드 1111 / 50에폭 =====")
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            [sys.executable, "-u", str(script_path), "--model", model_name,
             "--hidden", str(hidden), "--out_prefix", prefix],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    return_code = process.wait()
    print(f"\n{model_name} 종료 코드: {return_code} / 전체 실행 시간: {time.perf_counter() - started:.1f}초")
    outputs += [log_path, Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]
    if return_code != 0:
        raise RuntimeError(f"{model_name} 실행에 실패했습니다.")

backup_zip = log_dir / f"copy_memory_T1000_recurrent_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4

===== Copy Memory / T=1000 / GRU (hidden 61) / 시드 1111 / 50에폭 =====
Namespace(repo='/content/TCN', model='GRU', hidden=61, blank_len=1000, seq_len=10, epochs=50, batch_size=32, lr=0.0005, clip=1.0, seed=1111, out_prefix='/content/TCN/reproduction_logs/copy_memory_T1000_gru_seed1111_50epochs_20261124_053815')
MODEL_PARAMETERS: 12332

Test set: Average loss: 0.02438167  |  Accuracy: 99.1302
| Epoch   1 | last10 accuracy 11.2800 | last10 CE 2.21245646 | 1.9s

Test set: Average loss: 0.02134494  |  Accuracy: 99.1438
| Epoch   2 | last10 accuracy 12.6700 | last10 CE 2.10644722 | 1.3s

Test set: Average loss: 0.02069650  |  Accuracy: 99.1400
| Epoch   3 | last10 accuracy 12.2800 | last10 CE 2.08960986 | 1.3s

Test set: Average loss: 0.02050178  |  Accuracy: 99.1400
| Epoch   4 | last10 accuracy 12.2800 | last10 CE 2.08518195 | 1.3s

Test set: Average loss: 0.02044208  |  Accuracy: 99.1393
| Epoch   5 | last10 accuracy 12.2100 | l

In [8]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)

mine = Path("/content/tcn-reproduction")
if not mine.exists():
    subprocess.run(["git", "clone", "https://github.com/chamelewan/tcn-reproduction.git", str(mine)], check=True)
else:
    subprocess.run(["git", "-C", str(mine), "pull", "--ff-only"], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("내 저장소 커밋:", subprocess.check_output(["git", "-C", str(mine), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 500
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs = []


def run_and_log(title, cmd, log_path, cwd=None):
    print(f"\n===== {title} =====")
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    code = process.wait()
    print(f"\n종료 코드: {code} / 전체 실행 시간: {time.perf_counter() - started:.1f}초")
    if code != 0:
        raise RuntimeError(f"{title} 실행에 실패했습니다.")


# 1) TCN: 논문 부록 Table 2의 T=500 설정 (커널 6, 블록 9, 채널 10)
tcn_log = log_dir / f"copy_memory_T{T}_tcn_k6_l9_seed1111_50epochs_{stamp}.log"
tcn_ckpt = log_dir / f"copy_memory_T{T}_tcn_k6_l9_seed1111_50epochs_checkpoint.pt"
tcn_runner = f"""
import numpy as np, runpy, sys, torch
import torch.nn.functional as F
np.random.seed(1111)
sys.argv = ["copymem_test", "--blank_len", "{T}", "--seq_len", "10",
    "--ksize", "6", "--levels", "9", "--nhid", "10", "--dropout", "0.05",
    "--epochs", "50", "--seed", "1111", "--log-interval", "1000"]
state = runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")
model, test_x, test_y = state["model"], state["test_x"], state["test_y"]
model.eval()
with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    pred = logits.argmax(dim=-1)
    overall = (pred == test_y).float().mean().item() * 100
    last10 = (pred[:, -10:] == test_y[:, -10:]).float().mean().item() * 100
    last10_ce = F.cross_entropy(logits[:, -10:, :].reshape(-1, 10), test_y[:, -10:].reshape(-1)).item()
torch.save({{"model_state_dict": model.state_dict(), "completed_epochs": 50, "blank_len": {T},
            "ksize": 6, "levels": 9, "upstream_commit": "{commit}"}}, "{tcn_ckpt}")
print("SEQUENCE_TOTAL_LENGTH:", test_y.shape[1])
print("FINAL_MODEL_PARAMETERS:", sum(p.numel() for p in model.parameters()))
print(f"FINAL_OVERALL_ACCURACY: {{overall:.4f}}%")
print(f"FINAL_LAST10_ACCURACY: {{last10:.4f}}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {{last10_ce:.8f}}")
"""
run_and_log(f"Copy Memory / T={T} / TCN (k6, l9, h10) / 시드 1111 / 50에폭",
            [sys.executable, "-u", "-c", tcn_runner], tcn_log, cwd=str(repo))
outputs += [tcn_log, tcn_ckpt]

# 2) GRU·LSTM: T=1000과 같은 직접 구성 기준 모델, 길이만 500
script = mine / "scripts" / "run_copy_memory_recurrent.py"
for model_name, hidden in (("GRU", 61), ("LSTM", 53)):
    prefix = str(log_dir / f"copy_memory_T{T}_{model_name.lower()}_seed1111_50epochs_{stamp}")
    run_and_log(f"Copy Memory / T={T} / {model_name} (hidden {hidden}) / 시드 1111 / 50에폭",
                [sys.executable, "-u", str(script), "--repo", str(repo), "--model", model_name,
                 "--hidden", str(hidden), "--blank_len", str(T), "--out_prefix", prefix],
                Path(prefix + ".log"))
    outputs += [Path(prefix + ".log"), Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]

backup_zip = log_dir / f"copy_memory_T{T}_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
내 저장소 커밋: a141aeaf9eeb54814ae9d4650d84ae763b7e7f22
GPU: Tesla T4

===== Copy Memory / T=500 / TCN (k6, l9, h10) / 시드 1111 / 50에폭 =====
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=6, iters=100, levels=9, blank_len=500, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.04187851  |  Accuracy: 98.3550


Test set: Average loss: 0.02809098  |  Accuracy: 98.7521


Test set: Average loss: 0.01822623  |  Accuracy: 99.2677


Test set: Average loss: 0.02002030  |  Accuracy: 98.9292


Test set: Average loss: 0.01311785  |  Accuracy: 99.6437


Test set: Average loss: 0.01154705  |  Accuracy: 99.6983


Test set: Average l